# Submit MCTS — Connect Four (ml-arena)

Envoie l'agent **MCTS bitboard** (100 ms/coup) sur la compétition Connect Four.

> Ouvre en Colab, lance de haut en bas. Vérifie le token API et `COMPETITION_ID`
> (défaut `65` — **Connect Four**).
>
> Ne lance la cellule **Submit** qu'après le smoke test vert.


## 0. Setup


In [9]:
!pip install -q "mlarena-sdk==0.3.0" numpy pettingzoo

import mlarena

API_TOKEN = "mlk_user_f95442d5291b6917_cf7608823485d946ef5f7c38538fd66a"   # <-- Profile -> API Keys
COMPETITION_ID = 65

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")


## 1. Define your agent

La cellule suivante écrit **`agent.py`** (autonome : bitboard + MCTS). C'est ce
fichier que le worker exécute — pas d'import du dépôt local.


In [10]:
%%writefile agent.py
"""
MCTS Connect Four — flex_v1 contract (ml-arena).

flexkit's AEC loop drives your agent:
  1. Agent()                                   zero-arg constructor
  2. setup(observation_space, action_space)    once, before the first episode
  3. reset(env_player_name, episode_index)     at the start of EVERY episode
  4. choose_action(...); honor action_mask; return None once done

Self-contained: bitboard + MCTS, 100 ms/coup (limite challenge 250 ms).
"""
from __future__ import annotations

import gc
import math
import random
from time import perf_counter

import numpy as np

# ---------------------------------------------------------------------------
# Bitboard Connect Four
# ---------------------------------------------------------------------------
ROWS, COLS = 6, 7
H1 = ROWS + 1
SIZE = ROWS * COLS
COL_TOP = tuple(H1 * c + ROWS for c in range(COLS))
COL_BASE = tuple(H1 * c for c in range(COLS))
_EXPANSION_ORDER = (6, 0, 5, 1, 4, 2, 3)  # centre-first; pop() takes the end


def _won(bb: int) -> bool:
    m = bb & (bb >> 1)
    if m & (m >> 2):
        return True
    m = bb & (bb >> H1)
    if m & (m >> (2 * H1)):
        return True
    m = bb & (bb >> (H1 - 1))
    if m & (m >> (2 * (H1 - 1))):
        return True
    m = bb & (bb >> (H1 + 1))
    return bool(m & (m >> (2 * (H1 + 1))))


def _from_observation(observation):
    board = np.asarray(observation)
    mine, theirs = board[:, :, 0], board[:, :, 1]
    bb_turn = bb_other = 0
    heights = []
    moves = 0
    for c in range(COLS):
        base = COL_BASE[c]
        row = 0
        for r in range(ROWS - 1, -1, -1):
            if mine[r, c]:
                bb_turn |= 1 << (base + row)
            elif theirs[r, c]:
                bb_other |= 1 << (base + row)
            else:
                break
            row += 1
        heights.append(base + row)
        moves += row
    return bb_turn, bb_other, tuple(heights), moves


def _legal_columns(heights):
    return [c for c in range(COLS) if heights[c] < COL_TOP[c]]


def _play(state, col: int):
    bb_turn, bb_other, heights, moves = state
    pos = heights[col]
    bb_turn |= 1 << pos
    next_heights = list(heights)
    next_heights[col] = pos + 1
    moves += 1
    next_state = (bb_other, bb_turn, tuple(next_heights), moves)
    if _won(bb_turn):
        return next_state, -1.0
    if moves == SIZE:
        return next_state, 0.0
    return next_state, None


def _rollout(state) -> float:
    bb_turn, bb_other, heights, moves = state
    h = list(heights)
    playable = _legal_columns(h)
    randrange = random.randrange
    sign = 1.0
    while playable:
        i = randrange(len(playable))
        col = playable[i]
        pos = h[col]
        bb_turn |= 1 << pos
        pos += 1
        h[col] = pos
        if pos == COL_TOP[col]:
            playable[i] = playable[-1]
            playable.pop()
        moves += 1
        if _won(bb_turn):
            return sign
        if moves == SIZE:
            return 0.0
        bb_turn, bb_other = bb_other, bb_turn
        sign = -sign
    return 0.0


# ---------------------------------------------------------------------------
# MCTS
# ---------------------------------------------------------------------------
class _Node:
    __slots__ = (
        "state", "terminal_value", "action", "children", "untried", "visits", "value_sum",
    )

    def __init__(self, state, terminal_value, action=None):
        self.state = state
        self.terminal_value = terminal_value
        self.action = action
        self.children = []
        self.visits = 0
        self.value_sum = 0.0
        heights = state[2]
        self.untried = (
            []
            if terminal_value is not None
            else [c for c in _EXPANSION_ORDER if heights[c] < COL_TOP[c]]
        )

    def value_for_parent(self) -> float:
        return -self.value_sum / self.visits


def _uct_select(node: _Node, exploration_weight: float) -> _Node:
    log_parent = math.log(node.visits)
    best, best_score = None, float("-inf")
    for child in node.children:
        score = child.value_for_parent() + exploration_weight * math.sqrt(
            2 * log_parent / child.visits
        )
        if score > best_score:
            best, best_score = child, score
    assert best is not None
    return best


def _expand(node: _Node) -> _Node:
    action = node.untried.pop()
    state, terminal_value = _play(node.state, action)
    child = _Node(state, terminal_value, action=action)
    node.children.append(child)
    return child


def _simulate(root: _Node, exploration_weight: float, path: list) -> None:
    path.clear()
    path.append(root)
    node = root
    while node.terminal_value is None and not node.untried:
        node = _uct_select(node, exploration_weight)
        path.append(node)
    if node.terminal_value is None:
        node = _expand(node)
        path.append(node)
    value = (
        node.terminal_value
        if node.terminal_value is not None
        else _rollout(node.state)
    )
    for ancestor in reversed(path):
        ancestor.visits += 1
        ancestor.value_sum += value
        value = -value


def _mcts_search(state, n_sim, exploration_weight, time_budget):
    root = _Node(state, None)
    if not root.untried:
        return root, 0
    done = 0
    path: list = []
    if time_budget is None:
        while done < n_sim:
            _simulate(root, exploration_weight, path)
            done += 1
    else:
        deadline = perf_counter() + time_budget
        while perf_counter() < deadline:
            _simulate(root, exploration_weight, path)
            done += 1
    return root, done


def _best_action(root: _Node):
    if not root.children:
        return None
    return int(max(root.children, key=lambda c: (c.visits, c.value_for_parent())).action)


def _mcts_decide(observation, action_mask, n_sim, exploration_weight, time_budget):
    if time_budget is None:
        state = _from_observation(observation)
        root, done = _mcts_search(state, n_sim, exploration_weight, None)
        action = _best_action(root)
        if action is None:
            action = int(random.choice([i for i, ok in enumerate(action_mask) if ok]))
        return action, done

    collecting = gc.isenabled()
    if collecting:
        gc.disable()
    root = None
    try:
        deadline = perf_counter() + time_budget
        state = _from_observation(observation)
        remaining = deadline - perf_counter()
        root, done = _mcts_search(
            state, n_sim, exploration_weight, max(0.0, remaining)
        )
        action = _best_action(root)
        if action is None:
            action = int(random.choice([i for i, ok in enumerate(action_mask) if ok]))
        return action, done
    finally:
        root = None
        if collecting:
            gc.enable()


class Agent:
    """MCTS, 100 ms/coup par défaut (sous la limite challenge de 250 ms)."""

    def __init__(self):
        self.observation_space = None
        self.action_space = None
        self.env_player_name = ""
        self.episode_index = 0
        self.n_sim = 40
        self.time_budget = 0.1
        self.exploration_weight = 1.0
        self.last_simulations = 0

    def setup(self, observation_space, action_space):
        from flexkit.spaces import decode_space
        self.observation_space = decode_space(observation_space)
        self.action_space = decode_space(action_space)
        return True

    def reset(self, env_player_name, episode_index):
        self.env_player_name = env_player_name
        self.episode_index = episode_index
        return True

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        if action_mask is not None:
            legal = [i for i, ok in enumerate(action_mask) if ok]
            if not legal:
                return None
        action, self.last_simulations = _mcts_decide(
            observation,
            action_mask,
            self.n_sim,
            self.exploration_weight,
            self.time_budget,
        )
        return int(action)


Writing agent.py


## 2. Sanity-check local (avant submit)

Vérifie : coups légaux, gain immédiat, respect du budget 250 ms, une partie vs random.


In [11]:
import time

import numpy as np
from agent import Agent

CHALLENGE_LIMIT = 0.250  # secondes — contrainte ml-arena


def empty_obs():
    return np.zeros((6, 7, 2), dtype=np.int8)


def obs_with_columns(mine_cols, theirs_cols=()):
    """Construit une obs (6,7,2) en empilant des jetons depuis le bas."""
    board = np.zeros((6, 7, 2), dtype=np.int8)
    heights = [0] * 7
    # theirs d'abord puis mine, en alternant n'est pas requis ici : on pose
    # simplement les piles demandées (tests tactiques locaux).
    for col in theirs_cols:
        r = 5 - heights[col]
        board[r, col, 1] = 1
        heights[col] += 1
    for col in mine_cols:
        r = 5 - heights[col]
        board[r, col, 0] = 1
        heights[col] += 1
    return board


agent = Agent()
agent.reset("player_0", 0)
assert agent.time_budget == 0.1, agent.time_budget

# 1) plateau vide : coup légal + budget
mask = [1, 1, 1, 1, 1, 1, 1]
t0 = time.perf_counter()
action = agent.choose_action(empty_obs(), action_mask=mask)
dt = time.perf_counter() - t0
assert action in range(7), action
assert dt < CHALLENGE_LIMIT, f"trop lent: {dt*1000:.1f} ms"
print(f"vide → col={action}  {dt*1000:.1f} ms  sims={agent.last_simulations}")

# 2) colonne pleine exclue du mask
mask_full = [1, 1, 0, 1, 1, 1, 1]
action = agent.choose_action(empty_obs(), action_mask=mask_full)
assert action != 2 and action in (0, 1, 3, 4, 5, 6), action
print(f"mask sans col 2 → col={action}")

# 3) gain immédiat : 3 jetons à nous en bas de la col 3, jouer 3 doit gagner
#    plateau vu par nous : plan0 = nous. On a (3,0)/(3,1)/(3,2) à nous → jouer 3.
obs = obs_with_columns(mine_cols=[3, 3, 3])
mask = [1, 1, 1, 1, 1, 1, 1]
# budget court pour le test tactique (le submit garde 100 ms)
agent.time_budget = 0.05
t0 = time.perf_counter()
action = agent.choose_action(obs, action_mask=mask)
dt = time.perf_counter() - t0
assert action == 3, f"devait jouer le gain en 3, a joué {action}"
assert dt < CHALLENGE_LIMIT, dt
print(f"gain immédiat → col={action}  {dt*1000:.1f} ms  sims={agent.last_simulations}")
agent.time_budget = 0.1

# 4) une partie vs random (PettingZoo)
from pettingzoo.classic import connect_four_v3

env = connect_four_v3.env()
env.reset(seed=0)
names = list(env.agents)
me = Agent()
me.reset(names[0], 0)
reward = 0.0
for name in env.agent_iter():
    obs, rew, term, trunc, _ = env.last()
    if name == names[0]:
        reward = rew
    if term or trunc:
        env.step(None)
        continue
    if name == names[0]:
        env.step(me.choose_action(obs["observation"], rew, False, False, {}, obs["action_mask"]))
    else:
        legal = [i for i, ok in enumerate(obs["action_mask"]) if ok]
        env.step(int(np.random.default_rng(0).choice(legal)))
env.close()
print(f"1 partie vs random (seat0, seed0) → reward={reward:+.0f}")
print("smoke test OK — tu peux submit")


vide → col=3  101.7 ms  sims=3835
mask sans col 2 → col=3
gain immédiat → col=3  51.1 ms  sims=7504
1 partie vs random (seat0, seed0) → reward=+1
smoke test OK — tu peux submit


## 3. Submit


In [ ]:
# Upload + deploy. `tail_logs` spam `incomplete` pendant le run : on filtre.
result = client.submit(
    competition_id=COMPETITION_ID,
    files=["agent.py"],
    agent_name="MCTS-100ms",
)
aid = result["attache_agent_id"]
print("submitted:", result)

last_run_line = None
for line in client.tail_logs(COMPETITION_ID, aid, timeout_sec=900):
    # Évite de noyer la sortie avec le même "incomplete" toutes les 5 s.
    if line.startswith("  run:") and "outcome=incomplete" in line and "steps=None" in line:
        if line == last_run_line:
            continue
        last_run_line = line
    print(line)

status = client.agent_status(COMPETITION_ID, aid)
print("\n=== statut final ===")
print(status.get("status"), "-", status.get("last_status_message"))

games = client.agent_games(aid).get("games") or []
print(f"parties jouées au deploy: {len(games)}")
for g in games[:5]:
    perf = g.get("agent_performance") or {}
    print(
        f"  sim={g.get('simulation_id')}  outcome={perf.get('outcome')}  "
        f"reward={perf.get('reward')}  steps={g.get('env_nb_steps')}"
    )


## 4. Leaderboard


In [13]:
client.leaderboard(COMPETITION_ID)


,AgentName,AvatarKey,EloScore,EloVariance,FrontendPrecision,HasGpu,IsContinuous,IsEloRanked,IsMyAgent,LastRun,...,NumberOfRuns,Rank,RewardCi95,SimulationVersion,SubscriptionDate,TeamId,TeamMembers,TeamName,Username,agentAttachId
0,__benchmark__,shapes:77k1laip,1648,257.023194,4,False,False,True,False,2026-09-08 14:48:22,...,31,1,0.084570,flex_v1,2026-05-08 23:47:30,None,[],None,raphael,4569
1,agent,None,1216,15826.800000,4,False,False,True,False,2026-09-08 15:56:44,...,1,2,0.000000,flex_v1,2026-09-08 15:51:06,None,[],None,mat24.niz,8539
2,agent,None,1200,11155.560000,4,False,False,True,False,2026-09-08 14:48:14,...,2,3,0.401680,flex_v1,2026-09-08 14:13:09,None,[],None,amrouchean,8538
3,agent,None,1184,5570.442000,4,False,False,True,False,2026-09-08 14:48:14,...,4,4,0.290715,flex_v1,2026-09-08 11:56:47,None,[],None,mat24.niz,8537
4,Camus,shapes:77k1laip,1072,228.677158,4,False,False,True,False,2026-09-08 15:56:44,...,27,5,0.108840,flex_v1,2026-07-31 08:52:28,None,[],None,raphael,8489
5,flexfix-userB-ConnectFour,bottts:v1bu8r21,1056,204.048480,4,False,False,True,False,2026-09-08 14:48:28,...,28,6,0.100274,flex_v1,2026-07-04 11:27:40,None,[],None,jojo,8419
6,flexfix-userA-ConnectFour,shapes:77k1laip,1008,177.349006,4,False,False,True,False,2026-09-08 14:48:28,...,27,7,0.098362,flex_v1,2026-07-04 11:27:33,None,[],None,raphael,8413


## 5. Notes

- Budget défaut : **100 ms/coup** (limite challenge 250 ms).
- Fichier unique autonome — pas besoin du dépôt `projet` sur le worker.
- Toujours respecter `action_mask` (déjà filtré dans MCTS via les colonnes légales).
